# Parameter Grid Search

Grid search over signal parameters to find the IC/hit-rate-optimal configuration.

**This notebook does NOT run the backtester.** It only uses `analysis/signals.py` and `analysis/evaluation.py` on a representative sample period.

## Parameter Grid

| Parameter | Grid |
|---|---|
| `zscore_window_days` | [2, 5, 10] |
| `z_entry` | [1.5, 2.0, 2.5, 3.0] |
| `z_exit` | [0.0, 0.5, 1.0] |
| `z_stop` | [3.5, 4.0, 4.5] |
| `max_holding_minutes` | [60, 120, 240, 390] |
| `formation_window_days` | [42, 84, 126] |

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import itertools
import datetime as dt

import polars as pl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

from analysis.cointegration import find_cointegrated_pairs
from analysis.signals import compute_pvalue_weights, generate_pair_signals_for_day
from analysis.evaluation import evaluate_all
from analysis.preprocessing import load_processed
from utils.config import CONFIG, BARS_PER_DAY, MINUTES_PER_BAR, holding_horizons_bars

## Setup — Load Sample Period

We use a representative 3-month sample (Q3 2023) with 42 calendar days formation window.

In [ ]:
# Sample period for grid search (3 months of signal data)
SAMPLE_START = "2023-07-01"
SAMPLE_END   = "2023-09-30"
TIMEFRAME    = "15min"
BARS_PER_DAY_15 = BARS_PER_DAY[TIMEFRAME]  # 26

print(f"Signal evaluation window: {SAMPLE_START} → {SAMPLE_END}")
print(f"Bars per day at {TIMEFRAME}: {BARS_PER_DAY_15}")

In [ ]:
# Load intraday prices for all tickers over sample period (+ 10 extra days for forward returns)
# prices[ticker] = 15-min OHLCV DataFrame
from utils.config import get_all_tickers

tickers = get_all_tickers()
prices: dict[str, pl.DataFrame] = {}

for ticker in tickers:
    try:
        df = load_processed(ticker, TIMEFRAME, start_date=SAMPLE_START, end_date="2023-10-10")
        if df is not None and len(df) > 0:
            prices[ticker] = df
    except Exception:
        pass

print(f"Loaded price data for {len(prices)} tickers")

## Grid Search Loop

In [ ]:
import os
import multiprocessing as mp

# Parameter grids
GRID = {
    "zscore_window_days":    [2, 5, 10],
    "z_entry":               [1.5, 2.0, 2.5, 3.0],
    "z_exit":                [0.0, 0.5, 1.0],
    "z_stop":                [3.5, 4.0, 4.5],
    "max_holding_minutes":   [60, 120, 240, 390],
    "formation_window_days": [42, 84, 126],
}

# Holding horizons for evaluation (same as CONFIG defaults at 15min)
HOLDING_HORIZONS = holding_horizons_bars(TIMEFRAME)  # [4, 8, 16, 26]
print(f"Holding horizons (bars): {HOLDING_HORIZONS}")

# Total combinations
total = 1
for v in GRID.values():
    total *= len(v)
print(f"Total combinations: {total}")

# Workers: all CPUs minus 2, minimum 1
N_WORKERS = max(1, (os.cpu_count() or 4) - 2)
print(f"Worker processes: {N_WORKERS} (of {os.cpu_count()} CPUs)")

In [ ]:
import pandas_market_calendars as mcal

nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(start_date=SAMPLE_START, end_date=SAMPLE_END)
trading_days = [d.date() for d in schedule.index]
print(f"Trading days in sample: {len(trading_days)}")

In [ ]:
# ---------------------------------------------------------------------------
# Worker function
# Each worker receives one parameter combo and runs the full day loop.
# Defined at cell top-level so it's picklable under both "fork" and "spawn".
# "fork" is used below so child processes inherit `prices` and `trading_days`
# via copy-on-write — no data copying overhead.
# ---------------------------------------------------------------------------

def _eval_combo(args):
    """Evaluate one parameter combo. Returns a result dict or None."""
    (params, timeframe, bars_per_day_val, minutes_per_bar_val,
     holding_horizons, cost_bps) = args

    # Imports are already available via fork; spell them out for spawn compatibility
    import datetime as dt
    import polars as pl
    from analysis.cointegration import find_cointegrated_pairs
    from analysis.signals import compute_pvalue_weights, generate_pair_signals_for_day
    from analysis.evaluation import evaluate_all

    # These globals are inherited from the parent process via fork
    global prices, trading_days

    zscore_window_bars_val = params["zscore_window_days"] * bars_per_day_val
    max_holding_bars_val   = params["max_holding_minutes"] // minutes_per_bar_val
    formation_days         = params["formation_window_days"]

    all_day_signals: list[pl.DataFrame] = []

    for day in trading_days:
        formation_start = day - dt.timedelta(days=formation_days)

        try:
            pairs_df = find_cointegrated_pairs(
                start_date=str(formation_start),
                end_date=str(day - dt.timedelta(days=1)),
                timeframe=timeframe,
            )
        except Exception:
            continue

        if pairs_df is None or len(pairs_df) == 0:
            continue

        pairs_df = compute_pvalue_weights(pairs_df)

        pairs_df = pairs_df.filter(
            pl.col("ticker_a").is_in(list(prices.keys())) &
            pl.col("ticker_b").is_in(list(prices.keys()))
        )

        if len(pairs_df) == 0:
            continue

        day_prices = {
            t: df.filter(pl.col("timestamp").dt.date() == day)
            for t, df in prices.items()
        }

        try:
            signals_day = generate_pair_signals_for_day(
                pairs_df=pairs_df,
                date=day,
                intraday_prices=day_prices,
                zscore_window=zscore_window_bars_val,
                z_entry=params["z_entry"],
                z_exit=params["z_exit"],
                z_stop=params["z_stop"],
                max_holding_bars=max_holding_bars_val,
            )
        except Exception:
            continue

        if len(signals_day) > 0:
            all_day_signals.append(signals_day)

    if not all_day_signals:
        return None

    signals_all = pl.concat(all_day_signals)

    eval_df = evaluate_all(
        signals_df=signals_all,
        prices=prices,
        holding_bars_list=holding_horizons,
        cost_bps=cost_bps,
    )

    if len(eval_df) == 0:
        return None

    agg = eval_df.select([
        pl.col("ic_gross_weighted").mean(),
        pl.col("ic_net_weighted").mean(),
        pl.col("hit_rate_binary").mean(),
        pl.col("n_observations").sum(),
    ]).to_dicts()[0]

    return {
        **params,
        "ic_gross": agg["ic_gross_weighted"],
        "ic_net":   agg["ic_net_weighted"],
        "hit_rate": agg["hit_rate_binary"],
        "n_obs":    agg["n_observations"],
    }


# ---------------------------------------------------------------------------
# Build the argument list (one tuple per combo)
# ---------------------------------------------------------------------------

param_names  = list(GRID.keys())
param_values = list(GRID.values())

worker_args = [
    (
        dict(zip(param_names, combo)),
        TIMEFRAME,
        BARS_PER_DAY_15,
        MINUTES_PER_BAR[TIMEFRAME],
        HOLDING_HORIZONS,
        CONFIG.portfolio.transaction_cost_bps,
    )
    for combo in itertools.product(*param_values)
]

print(f"Dispatching {len(worker_args)} tasks across {N_WORKERS} workers...")

# ---------------------------------------------------------------------------
# Parallel execution — "fork" inherits prices + trading_days from parent
# ---------------------------------------------------------------------------

ctx = mp.get_context("fork")

results = []
n_done  = 0

with ctx.Pool(processes=N_WORKERS) as pool:
    for result in pool.imap_unordered(_eval_combo, worker_args):
        n_done += 1
        if result is not None:
            results.append(result)
        if n_done % 50 == 0 or n_done == len(worker_args):
            print(f"  {n_done}/{len(worker_args)} done  |  {len(results)} valid")

print(f"\nAll done. {len(results)} valid parameter combinations.")

## Results Table

In [ ]:
results_df = pl.DataFrame(results).sort("ic_net", descending=True)

print("Top 20 configurations by net IC:")
print(results_df.head(20).to_pandas().to_string(index=False))

In [ ]:
# Best by IC net
best_ic = results_df.row(0, named=True)
print("Best configuration by ic_net:")
for k, v in best_ic.items():
    print(f"  {k}: {v}")

# Best by hit rate
best_hit = results_df.sort("hit_rate", descending=True).row(0, named=True)
print("\nBest configuration by hit_rate:")
for k, v in best_hit.items():
    print(f"  {k}: {v}")

## Heatmaps

In [ ]:
# Heatmap: z_entry vs z_exit colored by IC net
fig, ax = plt.subplots(figsize=(8, 5))

pivot_ic = (
    results_df
    .group_by(["z_entry", "z_exit"])
    .agg(pl.col("ic_net").mean())
    .sort(["z_entry", "z_exit"])
    .to_pandas()
    .pivot(index="z_entry", columns="z_exit", values="ic_net")
)

vmax = max(abs(pivot_ic.values.min()), abs(pivot_ic.values.max()))
im = ax.imshow(pivot_ic.values, cmap="RdYlGn", vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_xticks(range(len(pivot_ic.columns)))
ax.set_xticklabels(pivot_ic.columns, rotation=0)
ax.set_yticks(range(len(pivot_ic.index)))
ax.set_yticklabels(pivot_ic.index)
ax.set_xlabel("z_exit")
ax.set_ylabel("z_entry")
ax.set_title("Mean Net IC: z_entry vs z_exit")
plt.colorbar(im, ax=ax)

for i in range(len(pivot_ic.index)):
    for j in range(len(pivot_ic.columns)):
        ax.text(j, i, f"{pivot_ic.values[i, j]:.3f}", ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: zscore_window_days vs formation_window_days colored by hit rate
fig, ax = plt.subplots(figsize=(7, 5))

pivot_hr = (
    results_df
    .group_by(["zscore_window_days", "formation_window_days"])
    .agg(pl.col("hit_rate").mean())
    .sort(["zscore_window_days", "formation_window_days"])
    .to_pandas()
    .pivot(index="zscore_window_days", columns="formation_window_days", values="hit_rate")
)

im = ax.imshow(pivot_hr.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(pivot_hr.columns)))
ax.set_xticklabels(pivot_hr.columns)
ax.set_yticks(range(len(pivot_hr.index)))
ax.set_yticklabels(pivot_hr.index)
ax.set_xlabel("formation_window_days")
ax.set_ylabel("zscore_window_days")
ax.set_title("Mean Hit Rate: zscore_window_days vs formation_window_days")
plt.colorbar(im, ax=ax)

for i in range(len(pivot_hr.index)):
    for j in range(len(pivot_hr.columns)):
        ax.text(j, i, f"{pivot_hr.values[i, j]:.3f}", ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## Best Configuration Summary

In [ ]:
print("=" * 60)
print("BEST CONFIGURATION BY NET IC")
print("=" * 60)
best_ic = results_df.sort("ic_net", descending=True).row(0, named=True)
for k, v in best_ic.items():
    if isinstance(v, float):
        print(f"  {k:30s}: {v:.4f}")
    else:
        print(f"  {k:30s}: {v}")

print()
print("=" * 60)
print("BEST CONFIGURATION BY HIT RATE")
print("=" * 60)
best_hr = results_df.sort("hit_rate", descending=True).row(0, named=True)
for k, v in best_hr.items():
    if isinstance(v, float):
        print(f"  {k:30s}: {v:.4f}")
    else:
        print(f"  {k:30s}: {v}")

In [ ]:
# Save results
import os

results_path = project_root / "results" / "evaluation" / "parameter_search.parquet"
results_path.parent.mkdir(parents=True, exist_ok=True)
results_df.write_parquet(results_path)
print(f"Results saved to {results_path}")